# Credit scoring v2 — draft

Application scorecard for the Tessera credit line product. **Draft, not reviewed.**

Unlike the fraud model this one is a regulated credit decision, so the bar is different:

- every decline needs **adverse-action reason codes** (ECOA / Reg B, FCRA §615)
- the model must be explainable to a customer in plain language, not just to us
- fair-lending review on protected classes before anything ships
- MRM sign-off under SR 11-7

It rides exactly the same rails as the fraud model: same feature module, same
`model-validation.yml` gate, same model card. The extra gates are additive.

In [1]:
import numpy as np
import pandas as pd

# Synthetic applications. No real applicant data in this repo, ever.
rng = np.random.default_rng(7)
n = 20_000
apps = pd.DataFrame({
    "application_id": [f"app_{i:06d}" for i in range(n)],
    "income_annual": rng.lognormal(10.9, 0.55, n).round(0),
    "utilization": rng.beta(2, 5, n).round(3),
    "months_since_delinquency": rng.integers(0, 96, n),
    "inquiries_6m": rng.poisson(1.3, n),
    "thin_file": (rng.random(n) < 0.18).astype(int),
})
print(apps.shape)
apps.head(3)

(20000, 6)


### TODO before this goes anywhere

- [ ] reason-code mapping from SHAP values → the 8 approved ECOA codes
- [ ] disparate-impact testing on the proxy-inferred groups (BISG), with an adverse-impact ratio floor
- [ ] stability: PSI against the booked population
- [ ] model card + MRM submission

None of this is done. The fraud model got there first; the plan is to reuse `ml/features/` and the validation workflow rather than build a second stack.